In [1]:
import os
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

def main():
    # Принудительно включаем однопоточный режим для сетевых запросов.
    # Это полностью исключает проблему бесконечного зависания (deadlock).
    dask.config.set(scheduler='single-threaded')

    output_dir = 'data'
    os.makedirs(output_dir, exist_ok=True)
    
    # Файл сохраняем с новым именем, чтобы не перезаписать тренировочные данные
    output_file = os.path.join(output_dir, 'era5_highres_test_2020.nc')

    # Путь к датасету высокого разрешения (0.25 градуса, 1440x721, шаг 6 часов)
    gcs_path = 'gs://weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr'
    
    print("Подключение к датасету WeatherBench 2...")
    ds = xr.open_zarr(gcs_path, consolidated=True, storage_options={'token': 'anon'})

    # Ограничиваемся 2020 годом и сразу берем непрерывный кусок: первые 128 срезов
    ds_2020 = ds.sel(time=slice('2020-01-01', '2020-12-31')).isel(time=slice(0, 128))

    # 8 наземных переменных
    surface_vars = [
        '2m_temperature',
        'mean_sea_level_pressure',
        '10m_u_component_of_wind',
        '10m_v_component_of_wind',
        'total_precipitation_6hr',
        'sea_surface_temperature',
        'total_column_water_vapour',
        'total_cloud_cover'
    ]

    # 5 атмосферных переменных
    atm_vars = [
        'temperature',
        'u_component_of_wind',
        'v_component_of_wind',
        'geopotential',
        'specific_humidity'
    ]
    
    # 4 слоя (итого 20 атмосферных каналов + 8 наземных = 28 признаков)
    levels = [1000, 925, 850, 700]

    print("Формирование графа задач (выбор переменных и слоев)...")
    ds_surface = ds_2020[surface_vars]
    ds_atm = ds_2020[atm_vars].sel(level=levels)
    
    # Объединяем наземные и атмосферные данные
    ds_final = xr.merge([ds_surface, ds_atm])

    print(f"Начинается скачивание и запись в {output_file}...")
    
    # Запускаем вычисление с отображением полосы прогресса
    with ProgressBar():
        ds_final.to_netcdf(output_file, compute=True)
        
    print("\nЗагрузка успешно завершена!")

if __name__ == "__main__":
    main()

Подключение к датасету WeatherBench 2...
Формирование графа задач (выбор переменных и слоев)...
Начинается скачивание и запись в data\era5_highres_test_2020.nc...
[########################################] | 100% Completed | 34m 46s

Загрузка успешно завершена!
